#Pipeline draft

#Getting data of all teams in the database

In [2]:
# Install dependencies as needed:

import kagglehub
from kagglehub import KaggleDatasetAdapter
from tabulate import tabulate

# Set the path to the file you'd like to load

# This  is just my exmaple for now
file_path = "base_data/teams.csv"

# Load the latest version
df_teams = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print(tabulate(df_teams.head(20), headers="keys", tablefmt='psql'))

100%|██████████| 508k/508k [00:01<00:00, 302kB/s]

+----+----------+-------------------------+-------------------------+----------------+-------------------------+--------------------+---------+------------------+-----------------------------------------------------+-----------+--------------------------+
|    |   teamId | location                | name                    | abbreviation   | displayName             | shortDisplayName   | color   | alternateColor   | logoURL                                             |   venueId | slug                     |
|----+----------+-------------------------+-------------------------+----------------+-------------------------+--------------------+---------+------------------+-----------------------------------------------------+-----------+--------------------------|
|  0 |        2 | Almagro                 | Almagro                 | ALM            | Almagro                 | Almagro            | 212121  | 000000           | https://a.espncdn.com/i/teamlogos/soccer/500/2.png  |      6130 | arg

#Getting all fixture data, and then only fixture data for prem teams.
##The purpose of this is so we can isolate prem teams from others, since there's no field like leagueID in the teams df that links them

In [3]:
file_path = "base_data/fixtures.csv"

df_fixtures = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

df_premfixtures = df_fixtures[df_fixtures["leagueId"] == 700]
# print(tabulate(df_premfixtures.head(20), headers="keys", tablefmt='psql'))

df_premfixtures["seasonType"].unique()

df_newseason = df_fixtures[df_fixtures["seasonType"] == 13481]
print(tabulate(df_premfixtures.head(100), headers="keys", tablefmt='psql'))



100%|██████████| 6.57M/6.57M [00:02<00:00, 2.43MB/s]


+-------+------+--------------+------------+-----------+---------------------+-----------+--------------+--------------+--------------+------------------+------------------+-----------------+-----------------+-------------------------+-------------------------+------------+---------------------+
|       |   Rn |   seasonType |   leagueId |   eventId | date                |   venueId |   attendance |   homeTeamId |   awayTeamId | homeTeamWinner   | awayTeamWinner   |   homeTeamScore |   awayTeamScore |   homeTeamShootoutScore |   awayTeamShootoutScore |   statusId | updateTime          |
|-------+------+--------------+------------+-----------+---------------------+-----------+--------------+--------------+--------------+------------------+------------------+-----------------+-----------------+-------------------------+-------------------------+------------+---------------------|
| 15375 |    1 |        12654 |        700 |    704279 | 2024-08-16 19:00:00 |       250 |        73297 |    

#Isolating prem teams IDs based on fixture dataframe

In [4]:
prem_teams_list = df_premfixtures["homeTeamId"].tolist()

#Keeping unique values only
prem_teams_ids = set(prem_teams_list)
print(prem_teams_ids)
print(len(prem_teams_ids))

df_premteams = df_teams[df_teams["teamId"].isin(prem_teams_ids)]

print(tabulate(df_premteams.head(20), headers="keys", tablefmt='psql'))

{384, 393, 331, 337, 349, 357, 359, 360, 361, 362, 363, 364, 366, 367, 368, 370, 371, 373, 375, 376, 379, 380, 382}
23
+-----+----------+-------------------------+-------------------------+----------------+-------------------------+--------------------+---------+------------------+------------------------------------------------------+-----------+-------------------+
|     |   teamId | location                | name                    | abbreviation   | displayName             | shortDisplayName   | color   | alternateColor   | logoURL                                              |   venueId | slug              |
|-----+----------+-------------------------+-------------------------+----------------+-------------------------+--------------------+---------+------------------+------------------------------------------------------+-----------+-------------------|
| 211 |      331 | Brighton & Hove Albion  | Brighton & Hove Albion  | BHA            | Brighton & Hove Albion  | Brighton      

#Getting the players, player_stats, teams and plays CSVs as dataframes

In [7]:
#Get players
file_path = "base_data/players.csv"

players_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get players stats
file_path = "playerStats_data/playerStats_2024_ENG.1.csv"

player_stats_2024_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get teams
file_path = "base_data/teams.csv"

teams_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get plays
file_path = "plays_data/plays_2024_ENG.1.csv"

plays_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

player_stats_2024_df.to_csv("players_stats.csv", sep=",")
print(tabulate(player_stats_2024_df.head(20), headers="keys", tablefmt='psql'))

/workspaces/Helix-Football-Data-App/myenvironment/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: nickName) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


+----+--------------+--------+----------+----------+-------------+---------------------+----------------+------------------------+-----------------------+---------------------+------------------+------------------+---------------------+------------------+-----------------------+--------------------+--------------------+--------------------+---------------+-----------------------+---------------------+
|    |   seasonType |   year | league   |   teamId |   athleteId |   appearances_value |   subIns_value |   foulsCommitted_value |   foulsSuffered_value |   yellowCards_value |   redCards_value |   ownGoals_value |   goalAssists_value |   offsides_value |   shotsOnTarget_value |   totalShots_value |   totalGoals_value |   shotsFaced_value |   saves_value |   goalsConceded_value | timestamp           |
|----+--------------+--------+----------+----------+-------------+---------------------+----------------+------------------------+-----------------------+---------------------+--------------